In [0]:
dbutils.widgets.text("catalog", "dev")
catalog = dbutils.widgets.get("catalog")

In [0]:
df_stores_bronze = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(f"/Volumes/{catalog}/landing/raw/stores/")
)
df_stores_bronze.display()

In [0]:
from pyspark.sql import functions as F

(
    df_stores_bronze
    .withColumn("source_filename", F.col("_metadata.file_name"))
    .withColumn("ingested_at", F.current_timestamp())
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{catalog}.bronze.stores")
)

## Ingestion to bronze customer table using COPY INTO

In [0]:
%sql
Create table if not exists identifier(:catalog).bronze.customers;

COPY INTO identifier(:catalog).bronze.customers
FROM (
  SELECT
    *,
    _metadata.file_name AS source_filename,
    current_timestamp() AS ingested_at
  FROM '/Volumes/fk_ce_2026_07_1363/landing/raw/customers/'
)
FILEFORMAT = csv
FORMAT_OPTIONS('header'='true', 'inferSchema'='true')
COPY_OPTIONS ('mergeSchema' = 'true')

## INgestion to bronze products 


In [0]:
df_products_bronze = (
    spark.read
    .format("json")
    .option("inferSchema", "true")
    .option("multiline", "true")
    .load(f"/Volumes/{catalog}/landing/raw/products/")
)

df_products_bronze.display()

In [0]:
(
    df_products_bronze
    .withColumn("source_filename", F.col("_metadata.file_name"))
    .withColumn("ingested_at", F.current_timestamp())
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{catalog}.bronze.products")
)

In [0]:
%sql
select * from identifier(:catalog).bronze.products

##Ingestion of orders using the autoloaders 


In [0]:

from pyspark.sql import functions as F

source_directory = f"/Volumes/{catalog}/landing/raw/orders/"
checkpoint_path = f"/Volumes/{catalog}/ops/ingestion/checkpoints/orders/"
schema_path    = f"/Volumes/{catalog}/ops/ingestion/schemas/orders/"

# Drop the corrupted table so Auto Loader re-ingests from scratch
# spark.sql(f"DROP TABLE IF EXISTS {catalog}.bronze.orders")

# Also clear any stale checkpoint from the previous (broken) run
dbutils.fs.rm(checkpoint_path, recurse=True)
dbutils.fs.rm(schema_path, recurse=True)

df_orders_bronze = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("multiline", "true")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("cloudFiles.rescuedDataColumn", "_rescued_data")
    .load(source_directory)
)

(
    df_orders_bronze
    .withColumn("source_filename", F.col("_metadata.file_name"))
    .withColumn("ingested_at", F.current_timestamp())
    .writeStream
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(f"{catalog}.bronze.orders")
)

In [0]:
%sql
select * from fk_ce_2026_07_1363.bronze.orders